# E2: Episodic Trace Memory on ALFWorld

### Conditions

| Condition | Description |
|-----------|-------------|
| **NoMemory** | Zero-shot |
| **RandomTrajectory** | k random expert trajectories |
| **FlatCosine** | k most similar by numpy cosine on task descriptions |
| **CVX-Episodic** | k most similar via CVX search, full episodes |
| **CVX-Causal** | Match ANY step via CVX, return only the **continuation** from match point |

### Key Hypothesis: CVX-Causal

Standard retrieval: *"Find tasks like mine, show me the full solution."*
Causal retrieval: *"Find a moment in any past episode where the agent was in a state like mine. Show me what they did NEXT."*

CVX-Causal searches across ALL action-observation steps (not just task descriptions).
When it finds a similar mid-episode state (e.g., "agent just picked up an apple"), it returns
the **continuation** — the steps that led to task completion from that point.
This is fundamentally impossible with flat cosine, which has no step ordering or episode structure.

**Dataset**: AgentInstruct ALFWorld (336 GPT-4 expert trajectories, 3–35 steps each)

In [1]:
import os, json, time, re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from scipy import stats

# === MODEL CONFIG ===
USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY'
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'
TOP_K = 3
N_EVAL = 100  # episodes to evaluate

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'Embedding: {EMBED_MODEL} (D={D})')
print(f'LLM: {LLM_MODEL} ({"Ollama" if USE_OLLAMA else "OpenAI"})')
print(f'Evaluation: n={N_EVAL} episodes, top-k={TOP_K}')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9111.13it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding: all-MiniLM-L6-v2 (D=384)
LLM: qwen2.5-coder:7b-instruct (Ollama)
Evaluation: n=100 episodes, top-k=3


## 1. Load AgentInstruct Trajectories

Expert ALFWorld trajectories generated by GPT-4.

In [2]:
from datasets import load_dataset
import re

# AgentInstruct: expert ALFWorld trajectories (GPT-4)
raw_ds = load_dataset('zai-org/agentinstruct', split='alfworld')
print(f'AgentInstruct alfworld: {len(raw_ds)} trajectories')

# Parse conversation format into structured episodes
# These are GPT-4 expert trajectories — all are successful demonstrations (reward=1.0)
ds = []
for row in raw_ds:
    convs = row['conversations']
    
    # Extract task from the human turn that describes the environment
    task = ''
    steps = []
    for turn in convs:
        text = turn['value']
        # Task is in the turn containing "Your task is to:"
        if 'Your task is to:' in text:
            match = re.search(r'Your task is to:\s*(.+)', text)
            if match:
                task = match.group(1).strip()
        # GPT actions are the agent steps
        elif turn['from'] == 'gpt' and ('ACTION:' in text or 'THOUGHT:' in text):
            action_match = re.search(r'ACTION:\s*(.+)', text)
            action = action_match.group(1).strip() if action_match else ''
            thought_match = re.search(r'THOUGHT:\s*(.+?)(?:\n|ACTION:)', text, re.DOTALL)
            thought = thought_match.group(1).strip() if thought_match else ''
            if action:
                steps.append({'action': action, 'observation': thought})
        # Human feedback (environment response) — attach to last step
        elif turn['from'] == 'human' and len(steps) > 0 and 'Your task is to:' not in text:
            steps[-1]['observation'] = text[:200]
    
    if task and steps:
        # AgentInstruct trajectories are expert demonstrations — all successful
        reward = 1.0
        
        # Infer task type from task description
        task_lower = task.lower()
        if 'clean' in task_lower: task_type = 'clean'
        elif 'heat' in task_lower: task_type = 'heat'
        elif 'cool' in task_lower: task_type = 'cool'
        elif 'examine' in task_lower or 'look' in task_lower: task_type = 'examine'
        elif 'put' in task_lower or 'find' in task_lower: task_type = 'put'
        else: task_type = 'pick'
        
        ds.append({
            'task': task,
            'steps': steps,
            'reward': reward,
            'task_type': task_type,
        })

print(f'Parsed {len(ds)} episodes from AgentInstruct')
n_success = sum(1 for e in ds if e['reward'] >= 0.7)
print(f'  Successful: {n_success}/{len(ds)} (expert demonstrations)')
print(f'  Task types: {dict(pd.Series([e["task_type"] for e in ds]).value_counts())}')
print(f'  Steps/episode: min={min(len(e["steps"]) for e in ds)}, '
      f'median={np.median([len(e["steps"]) for e in ds]):.0f}, '
      f'max={max(len(e["steps"]) for e in ds)}')
print(f'\nSample task: {ds[0]["task"][:80]}')
print(f'Sample steps ({len(ds[0]["steps"])}):')
for s in ds[0]['steps'][:3]:
    print(f'  Action: {s["action"][:60]}')
    print(f'  Obs: {s["observation"][:60]}')

AgentInstruct alfworld: 336 trajectories
Parsed 336 episodes from AgentInstruct
  Successful: 336/336 (expert demonstrations)
  Task types: {'put': np.int64(158), 'clean': np.int64(68), 'cool': np.int64(49), 'examine': np.int64(37), 'heat': np.int64(24)}
  Steps/episode: min=3, median=10, max=35

Sample task: find two laptop and put them in bed.
Sample steps (14):
  Action: go to diningtable 1
  Obs: On the diningtable 1, you see a alarmclock 2, a bowl 2, a cd
  Action: take laptop 1 from diningtable 1
  Obs: You pick up the laptop 1 from the diningtable 1.
  Action: go to bed 1
  Obs: On the bed 1, you see a pillow 2, and a pillow 1.


## 2. Build Episodic Index

In [3]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'alfworld_episodic.cvx')
FLAT_EMB_PATH = str(DATA_DIR / 'alfworld_task_embeddings.npy')

def encode_entity(episode_id, step_index):
    return (episode_id << 16) | step_index

if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded: {len(index)} points')
else:
    print('Building episodic memory...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    episode_data = {}
    
    for ep_idx, traj in enumerate(ds):
        task = traj['task']
        steps = traj['steps']
        reward = traj['reward']
        
        for s_idx, step in enumerate(steps):
            text = f"Task: {task} | Action: {step.get('action', '')} | Obs: {step.get('observation', '')}"
            vec = embedder.encode(text[:500]).tolist()
            eid = encode_entity(ep_idx, s_idx)
            index.insert(entity_id=eid, timestamp=ep_idx * 10000 + s_idx, vector=vec)
        
        episode_data[ep_idx] = {
            'task': task,
            'steps': steps,
            'reward': reward,
            'n_steps': len(steps),
            'task_type': traj.get('task_type', 'unknown'),
        }
    
    index.save(INDEX_PATH)
    with open(str(DATA_DIR / 'alfworld_metadata.json'), 'w') as f:
        json.dump(episode_data, f)
    print(f'Built: {len(index)} points ({len(episode_data)} episodes)')

with open(str(DATA_DIR / 'alfworld_metadata.json')) as f:
    episode_data = {int(k): v for k, v in json.load(f).items()}

# Build flat embedding matrix for cosine baseline (task descriptions only)
if os.path.exists(FLAT_EMB_PATH):
    task_embeddings = np.load(FLAT_EMB_PATH)
    print(f'Loaded flat task embeddings: {task_embeddings.shape}')
else:
    print('Building flat task embeddings...')
    task_embeddings = embedder.encode([episode_data[i]['task'] for i in range(len(episode_data))])
    np.save(FLAT_EMB_PATH, task_embeddings)
    print(f'Built flat embeddings: {task_embeddings.shape}')

n_success = sum(1 for e in episode_data.values() if e['reward'] >= 0.7)
print(f'{len(episode_data)} episodes ({n_success} successful), flat matrix {task_embeddings.shape}')

Loaded: 4542 points
Loaded flat task embeddings: (336, 384)
336 episodes (336 successful), flat matrix (336, 384)


## 3. Retrieval & Evaluation

In [4]:
def retrieve_cvx(task_description, top_k=TOP_K, min_reward=0.7, exclude_ep=None):
    """Retrieve similar episodes via CVX temporal search (step_0 only = task match)."""
    query_vec = embedder.encode(f'Task: {task_description}').tolist()
    results = index.search(vector=query_vec, k=top_k * 20)
    
    seen = set()
    episodes = []
    for node_id, ts, score in results:
        ep_idx = ts // 10000
        if ep_idx in seen or ep_idx not in episode_data:
            continue
        if exclude_ep is not None and ep_idx == exclude_ep:
            continue
        ep = episode_data[ep_idx]
        if ep['reward'] < min_reward:
            continue
        seen.add(ep_idx)
        episodes.append({'episode_id': ep_idx, 'similarity': score, **ep})
        if len(episodes) >= top_k:
            break
    return episodes


def retrieve_causal_cvx(task_description, top_k=TOP_K, exclude_ep=None):
    """CVX-Causal: search ALL action-observation steps, return CONTINUATION from match.
    
    This searches the full step embedding space (not just task descriptions).
    When it matches step N of episode E, it returns steps N+1..end: the resolution
    path from a similar mid-episode state.
    
    This is the killer feature: "I'm in a state where I just picked up an apple.
    What did successful agents do next?" → returns the remaining steps to completion.
    """
    query_vec = embedder.encode(f'Task: {task_description}').tolist()
    results = index.search(vector=query_vec, k=top_k * 30)
    
    seen = set()
    episodes = []
    for node_id, ts, score in results:
        ep_idx = ts // 10000
        match_step = ts % 10000
        
        if ep_idx in seen or ep_idx not in episode_data:
            continue
        if exclude_ep is not None and ep_idx == exclude_ep:
            continue
        
        ep = episode_data[ep_idx]
        seen.add(ep_idx)
        
        # Extract continuation: steps AFTER the match point
        all_steps = ep['steps']
        continuation = all_steps[match_step + 1:] if match_step + 1 < len(all_steps) else all_steps[-3:]
        
        episodes.append({
            'episode_id': ep_idx,
            'match_step': match_step,
            'total_steps': ep['n_steps'],
            'continuation': continuation,
            'similarity': score,
            **ep,
        })
        if len(episodes) >= top_k:
            break
    return episodes


def retrieve_flat_cosine(task_description, top_k=TOP_K, exclude_ep=None):
    """Flat numpy cosine on task descriptions only."""
    query_vec = embedder.encode(task_description)
    query_norm = query_vec / (np.linalg.norm(query_vec) + 1e-8)
    emb_norms = task_embeddings / (np.linalg.norm(task_embeddings, axis=1, keepdims=True) + 1e-8)
    sims = emb_norms @ query_norm
    
    top_indices = np.argsort(sims)[::-1]
    episodes = []
    for idx in top_indices:
        idx = int(idx)
        if exclude_ep is not None and idx == exclude_ep:
            continue
        if idx in episode_data:
            episodes.append({'episode_id': idx, 'similarity': float(sims[idx]), **episode_data[idx]})
        if len(episodes) >= top_k:
            break
    return episodes


def retrieve_random(top_k=TOP_K, exclude_ep=None):
    """Random baseline."""
    candidates = [i for i in range(len(episode_data)) if i != exclude_ep]
    indices = np.random.choice(candidates, size=top_k, replace=False)
    return [{'episode_id': int(i), **episode_data[int(i)]} for i in indices]


def format_trajectory(episodes):
    """Standard format: full episode trajectory."""
    lines = []
    for i, ep in enumerate(episodes, 1):
        lines.append(f'[Past Episode {i} — Success, {ep["n_steps"]} steps]')
        lines.append(f'Task: {ep["task"]}')
        for j, step in enumerate(ep['steps'][:12], 1):
            if isinstance(step, dict):
                lines.append(f'  Step {j}: {step.get("action", "")} → {step.get("observation", "")[:80]}')
            else:
                lines.append(f'  Step {j}: {str(step)[:100]}')
        lines.append('')
    return '\n'.join(lines)


def format_causal_trajectory(episodes):
    """Causal format: show only the CONTINUATION from match point.
    
    Instead of "here's a full episode for a similar task", this says:
    "An agent was in a state similar to yours at step N. Here's what they did next to complete the task."
    """
    lines = []
    for i, ep in enumerate(episodes, 1):
        match_step = ep.get('match_step', 0)
        total = ep.get('total_steps', ep['n_steps'])
        continuation = ep.get('continuation', ep['steps'])
        
        lines.append(f'[From a similar state (step {match_step}/{total} of a past episode)]')
        lines.append(f'Original task: {ep["task"]}')
        lines.append(f'From this point, the agent completed the task in {len(continuation)} more steps:')
        for j, step in enumerate(continuation[:12], 1):
            if isinstance(step, dict):
                lines.append(f'  → {step.get("action", "")}')
            else:
                lines.append(f'  → {str(step)[:100]}')
        lines.append('')
    return '\n'.join(lines)


# === EVALUATION METRICS ===
ACTION_VERBS = {'go', 'take', 'put', 'open', 'close', 'use', 'examine', 'look', 'clean', 'heat', 'cool', 'toggle'}

def extract_actions(text):
    words = set(text.lower().split())
    return words & ACTION_VERBS

def extract_objects(text):
    return set(re.findall(r'[a-z]+(?:table|shelf|counter|fridge|microwave|sink|toilet|drawer|cabinet|desk|bed|sofa|armchair|lamp|stove)\s*\d*', text.lower()))

def compute_plan_metrics(plan_text, expert_episode):
    expert_text = ' '.join(
        step.get('action', '') + ' ' + step.get('observation', '')
        for step in expert_episode['steps'] if isinstance(step, dict)
    )
    
    plan_steps = len([l for l in plan_text.strip().split('\n') if l.strip()])
    step_ratio = plan_steps / max(expert_episode['n_steps'], 1)
    
    plan_actions = extract_actions(plan_text)
    expert_actions = extract_actions(expert_text)
    action_overlap = len(plan_actions & expert_actions) / max(len(expert_actions), 1)
    
    plan_objects = extract_objects(plan_text)
    expert_objects = extract_objects(expert_text)
    object_recall = len(plan_objects & expert_objects) / max(len(expert_objects), 1)
    
    plan_emb = embedder.encode(plan_text[:500])
    expert_emb = embedder.encode(expert_text[:500])
    cos_sim = float(np.dot(plan_emb, expert_emb) / (np.linalg.norm(plan_emb) * np.linalg.norm(expert_emb) + 1e-8))
    
    return {
        'n_steps': plan_steps,
        'step_ratio': step_ratio,
        'action_overlap': action_overlap,
        'object_recall': object_recall,
        'semantic_sim': cos_sim,
    }


# Test all retrieval methods
test_task = episode_data[0]['task']
print(f'Query: {test_task}\n')
for name, fn in [('CVX-Episodic', lambda: retrieve_cvx(test_task, exclude_ep=0)),
                 ('FlatCosine', lambda: retrieve_flat_cosine(test_task, exclude_ep=0)),
                 ('CVX-Causal', lambda: retrieve_causal_cvx(test_task, exclude_ep=0))]:
    eps = fn()
    print(f'{name}: {len(eps)} episodes')
    for ep in eps:
        step_info = f' (matched step {ep["match_step"]}/{ep["total_steps"]}, cont={len(ep["continuation"])} steps)' if 'match_step' in ep else ''
        print(f'  [{ep["episode_id"]}] sim={ep["similarity"]:.3f}{step_info}: {ep["task"][:50]}')
    print()

Query: find two laptop and put them in bed.



CVX-Episodic: 3 episodes
  [259] sim=0.307: put two laptop in bed.
  [45] sim=0.493: find two book and put them in bed.
  [86] sim=0.520: put two book in bed.



FlatCosine: 3 episodes
  [259] sim=0.943: put two laptop in bed.
  [45] sim=0.673: find two book and put them in bed.
  [55] sim=0.654: put two book in bed.

CVX-Causal: 3 episodes
  [259] sim=0.307 (matched step 6/8, cont=1 steps): put two laptop in bed.
  [45] sim=0.493 (matched step 6/17, cont=10 steps): find two book and put them in bed.
  [86] sim=0.520 (matched step 2/8, cont=5 steps): put two book in bed.



In [5]:
def solve_with_llm(task, context=''):
    """Ask LLM to produce a plan for the task."""
    system = (
        'You are an embodied agent in a household. Given a task, produce a step-by-step '
        'action plan. Each step should be one action like "go to countertop 1", "take apple 1 from countertop 1", '
        '"put apple 1 in/on fridge 1". Use the exact ALFWorld action format. '
        'Output ONLY the numbered steps, no explanation.'
    )
    user_parts = []
    if context:
        user_parts.append('Here are examples of how similar tasks were solved:\n')
        user_parts.append(context)
        user_parts.append('\n---\nNow solve this task:\n')
    user_parts.append(f'Task: {task}')
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=0.0,
        max_tokens=512,
    )
    return response.choices[0].message.content


# Stratified sample
task_type_counts = pd.Series([episode_data[i]['task_type'] for i in range(len(episode_data))]).value_counts()
eval_indices = []
np.random.seed(42)
for tt, count in task_type_counts.items():
    tt_indices = [i for i in range(len(episode_data)) if episode_data[i]['task_type'] == tt]
    n_sample = max(1, int(N_EVAL * count / len(episode_data)))
    eval_indices.extend(np.random.choice(tt_indices, size=min(n_sample, len(tt_indices)), replace=False))
eval_indices = eval_indices[:N_EVAL]
print(f'Evaluation: {len(eval_indices)} episodes (stratified)')
print(f'  Types: {dict(pd.Series([episode_data[i]["task_type"] for i in eval_indices]).value_counts())}')

# Conditions (with leave-one-out)
def get_context_nomemory(task, ep_idx, k):
    return ''

def get_context_random(task, ep_idx, k):
    return format_trajectory(retrieve_random(top_k=k, exclude_ep=ep_idx))

def get_context_flat(task, ep_idx, k):
    return format_trajectory(retrieve_flat_cosine(task, top_k=k, exclude_ep=ep_idx))

def get_context_cvx(task, ep_idx, k):
    return format_trajectory(retrieve_cvx(task, top_k=k, exclude_ep=ep_idx))

def get_context_causal(task, ep_idx, k):
    return format_causal_trajectory(retrieve_causal_cvx(task, top_k=k, exclude_ep=ep_idx))

conditions = {
    'NoMemory': get_context_nomemory,
    'RandomTrajectory': get_context_random,
    'FlatCosine': get_context_flat,
    'CVX-Episodic': get_context_cvx,
    'CVX-Causal': get_context_causal,
}

# Run evaluation
all_results = {cond: [] for cond in conditions}
t0 = time.perf_counter()

for i, ep_idx in enumerate(eval_indices):
    ep = episode_data[ep_idx]
    task = ep['task']
    
    for cond_name, get_ctx in conditions.items():
        context = get_ctx(task, ep_idx, TOP_K)
        plan = solve_with_llm(task, context)
        metrics = compute_plan_metrics(plan, ep)
        metrics['task_type'] = ep['task_type']
        metrics['ep_idx'] = ep_idx
        all_results[cond_name].append(metrics)
    
    if (i + 1) % 20 == 0:
        elapsed = time.perf_counter() - t0
        eta = elapsed / (i + 1) * (len(eval_indices) - i - 1)
        print(f'[{i+1}/{len(eval_indices)}] elapsed={elapsed:.0f}s, ETA={eta:.0f}s')

print(f'\nDone in {time.perf_counter() - t0:.0f}s')

# Aggregate
print('\n=== OVERALL RESULTS ===')
print(f'{"Condition":>18} | {"SemSim":>7} | {"ActOvl":>7} | {"ObjRec":>7} | {"StepR":>6}')
print('-' * 55)
for cond, res in all_results.items():
    df_r = pd.DataFrame(res)
    print(f'{cond:>18} | {df_r["semantic_sim"].mean():>7.3f} | {df_r["action_overlap"].mean():>7.2f} | '
          f'{df_r["object_recall"].mean():>7.2f} | {df_r["step_ratio"].mean():>6.2f}')

Evaluation: 99 episodes (stratified)
  Types: {'put': np.int64(47), 'clean': np.int64(20), 'cool': np.int64(14), 'examine': np.int64(11), 'heat': np.int64(7)}


[20/99] elapsed=186s, ETA=733s


[40/99] elapsed=351s, ETA=517s


[60/99] elapsed=521s, ETA=339s


[80/99] elapsed=682s, ETA=162s



Done in 819s

=== OVERALL RESULTS ===
         Condition |  SemSim |  ActOvl |  ObjRec |  StepR
-------------------------------------------------------
          NoMemory |   0.600 |    0.60 |    0.09 |   0.56
  RandomTrajectory |   0.614 |    0.78 |    0.17 |   0.70
        FlatCosine |   0.702 |    0.89 |    0.30 |   0.74
      CVX-Episodic |   0.709 |    0.88 |    0.28 |   0.74
        CVX-Causal |   0.588 |    0.70 |    0.21 |   0.43


In [6]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

colors = {
    'NoMemory': '#95a5a6', 'RandomTrajectory': '#3498db', 'FlatCosine': '#f39c12',
    'CVX-Episodic': '#e74c3c', 'CVX-Causal': '#2ecc71',
}

# Multi-metric bar chart
metrics_to_plot = ['semantic_sim', 'action_overlap', 'object_recall', 'step_ratio']
metric_labels = ['Semantic Similarity', 'Action Verb Overlap', 'Object Recall', 'Step Ratio']

fig = make_subplots(rows=2, cols=2, subplot_titles=metric_labels)

for idx, (metric, label) in enumerate(zip(metrics_to_plot, metric_labels)):
    row, col = idx // 2 + 1, idx % 2 + 1
    for cond in conditions:
        vals = [r[metric] for r in all_results[cond]]
        fig.add_trace(go.Bar(
            x=[cond], y=[np.mean(vals)],
            error_y=dict(type='data', array=[np.std(vals)], visible=True),
            marker_color=colors[cond],
            name=cond, showlegend=(idx == 0),
        ), row=row, col=col)

fig.update_layout(height=600, width=900, title=f'Plan Quality Metrics (n={len(eval_indices)}, k={TOP_K})')
fig.show()

# Task-type breakdown for semantic similarity
fig2 = go.Figure()
task_types = sorted(set(r['task_type'] for r in all_results['NoMemory']))

for cond in conditions:
    means = []
    for tt in task_types:
        vals = [r['semantic_sim'] for r in all_results[cond] if r['task_type'] == tt]
        means.append(np.mean(vals) if vals else 0)
    fig2.add_trace(go.Bar(x=task_types, y=means, name=cond, marker_color=colors[cond]))

fig2.update_layout(
    title='Semantic Similarity by Task Type',
    xaxis_title='Task Type', yaxis_title='Cosine Similarity',
    barmode='group', height=400, width=800,
)
fig2.show()

## 4. Statistical Tests

In [7]:
# Wilcoxon signed-rank tests
print('=== Wilcoxon Signed-Rank Tests (semantic similarity) ===\n')

pairs = [
    ('CVX-Causal', 'NoMemory'),
    ('CVX-Causal', 'FlatCosine'),
    ('CVX-Causal', 'CVX-Episodic'),
    ('CVX-Causal', 'RandomTrajectory'),
    ('CVX-Episodic', 'FlatCosine'),
    ('FlatCosine', 'NoMemory'),
]

for a, b in pairs:
    a_vals = np.array([r['semantic_sim'] for r in all_results[a]])
    b_vals = np.array([r['semantic_sim'] for r in all_results[b]])
    diff = a_vals - b_vals
    nonzero = diff[diff != 0]
    if len(nonzero) > 0:
        stat, p_val = stats.wilcoxon(nonzero, alternative='greater')
    else:
        stat, p_val = 0, 1.0
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'{a} vs {b}: Δ={np.mean(diff):+.4f}, W={stat:.0f}, p={p_val:.4f} {sig}')

print('\n=== Wilcoxon (object recall) ===\n')
for a, b in pairs:
    a_vals = np.array([r['object_recall'] for r in all_results[a]])
    b_vals = np.array([r['object_recall'] for r in all_results[b]])
    diff = a_vals - b_vals
    nonzero = diff[diff != 0]
    if len(nonzero) > 0:
        stat, p_val = stats.wilcoxon(nonzero, alternative='greater')
    else:
        stat, p_val = 0, 1.0
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'{a} vs {b}: Δ={np.mean(diff):+.4f}, W={stat:.0f}, p={p_val:.4f} {sig}')

# CVX-Causal step distribution
print('\n=== CVX-Causal: Match Step Distribution ===')
step_counts = {}
for ep_idx in eval_indices:
    task = episode_data[ep_idx]['task']
    eps = retrieve_causal_cvx(task, top_k=TOP_K, exclude_ep=ep_idx)
    for ep in eps:
        ms = ep.get('match_step', 0)
        step_counts[ms] = step_counts.get(ms, 0) + 1
total = sum(step_counts.values())
for step in sorted(step_counts.keys())[:10]:
    pct = step_counts[step] / total
    bar = '█' * int(pct * 40)
    print(f'  Step {step:>2}: {step_counts[step]:>4} ({pct:.1%}) {bar}')
if len(step_counts) > 10:
    remaining = sum(v for k, v in step_counts.items() if k >= 10)
    print(f'  Step 10+: {remaining:>4} ({remaining/total:.1%})')

# Retrieval overlap
print('\n=== Retrieval Overlap ===')
for a_name, a_fn, b_name, b_fn in [
    ('CVX-Causal', lambda t, e: retrieve_causal_cvx(t, exclude_ep=e), 'CVX-Episodic', lambda t, e: retrieve_cvx(t, exclude_ep=e)),
    ('CVX-Causal', lambda t, e: retrieve_causal_cvx(t, exclude_ep=e), 'FlatCosine', lambda t, e: retrieve_flat_cosine(t, exclude_ep=e)),
]:
    overlaps = []
    for ep_idx in eval_indices:
        task = episode_data[ep_idx]['task']
        a_eps = a_fn(task, ep_idx)
        b_eps = b_fn(task, ep_idx)
        a_ids = {ep['episode_id'] for ep in a_eps}
        b_ids = {ep['episode_id'] for ep in b_eps}
        overlaps.append(len(a_ids & b_ids) / TOP_K if TOP_K > 0 else 0)
    print(f'{a_name} vs {b_name}: {np.mean(overlaps):.1%} overlap')

# Final summary
print('\n=== FINAL SUMMARY ===')
print(f'{"Condition":>18} | {"SemSim":>7} | {"ActOvl":>7} | {"ObjRec":>7} | {"StepR":>6}')
print('-' * 55)
for cond in conditions:
    df_r = pd.DataFrame(all_results[cond])
    print(f'{cond:>18} | {df_r["semantic_sim"].mean():>7.3f} | {df_r["action_overlap"].mean():>7.2f} | '
          f'{df_r["object_recall"].mean():>7.2f} | {df_r["step_ratio"].mean():>6.2f}')
print(f'\nModel: {LLM_MODEL}, Dataset: {len(episode_data)} episodes, Eval: {len(eval_indices)} (LOO), k={TOP_K}')

=== Wilcoxon Signed-Rank Tests (semantic similarity) ===

CVX-Causal vs NoMemory: Δ=-0.0122, W=2059, p=0.9267 ns
CVX-Causal vs FlatCosine: Δ=-0.1142, W=559, p=1.0000 ns
CVX-Causal vs CVX-Episodic: Δ=-0.1213, W=485, p=1.0000 ns
CVX-Causal vs RandomTrajectory: Δ=-0.0263, W=1931, p=0.9712 ns
CVX-Episodic vs FlatCosine: Δ=+0.0071, W=1090, p=0.2131 ns
FlatCosine vs NoMemory: Δ=+0.1020, W=4434, p=0.0000 ***

=== Wilcoxon (object recall) ===

CVX-Causal vs NoMemory: Δ=+0.1170, W=151, p=0.0002 ***
CVX-Causal vs FlatCosine: Δ=-0.0909, W=24, p=0.9922 ns
CVX-Causal vs CVX-Episodic: Δ=-0.0749, W=38, p=0.9730 ns
CVX-Causal vs RandomTrajectory: Δ=+0.0354, W=26, p=0.0234 *
CVX-Episodic vs FlatCosine: Δ=-0.0160, W=4, p=0.8438 ns
FlatCosine vs NoMemory: Δ=+0.2079, W=413, p=0.0000 ***

=== CVX-Causal: Match Step Distribution ===


  Step  0:   21 (7.1%) ██
  Step  1:    8 (2.7%) █
  Step  2:    9 (3.0%) █
  Step  3:   22 (7.4%) ██
  Step  4:   15 (5.1%) ██
  Step  5:   23 (7.7%) ███
  Step  6:   25 (8.4%) ███
  Step  7:   39 (13.1%) █████
  Step  8:    9 (3.0%) █
  Step  9:   17 (5.7%) ██
  Step 10+:  109 (36.7%)

=== Retrieval Overlap ===


CVX-Causal vs CVX-Episodic: 94.3% overlap


CVX-Causal vs FlatCosine: 67.3% overlap

=== FINAL SUMMARY ===
         Condition |  SemSim |  ActOvl |  ObjRec |  StepR
-------------------------------------------------------
          NoMemory |   0.600 |    0.60 |    0.09 |   0.56
  RandomTrajectory |   0.614 |    0.78 |    0.17 |   0.70
        FlatCosine |   0.702 |    0.89 |    0.30 |   0.74
      CVX-Episodic |   0.709 |    0.88 |    0.28 |   0.74
        CVX-Causal |   0.588 |    0.70 |    0.21 |   0.43

Model: qwen2.5-coder:7b-instruct, Dataset: 336 episodes, Eval: 99 (LOO), k=3
